# Credit Card Fraud Detection

End-to-end ML project to detect fraudulent transactions. Trained and compared 6 classifiers (Logistic Regression, Random Forest, XGBoost, LightGBM, CatBoost, SVM) on the Kaggle Credit Card Fraud dataset with SMOTE for class imbalance.

**Dataset:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Goal:** Identify fraudulent transactions with high recall while keeping false positives low.

---

## Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay, RocCurveDisplay
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import joblib
import os, time

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", None)

print("all imports done")

## Data Loading

In [ ]:
DATA_PATH = "creditcard.csv"

if not os.path.exists(DATA_PATH):
    print("downloading dataset...")
    import requests
    r = requests.get("https://raw.githubusercontent.com/nsethi31/Kaggle-Credit-Card-Fraud-Detection/master/creditcard.csv")
    with open(DATA_PATH, "wb") as f:
        f.write(r.content)
    print("done")

df = pd.read_csv(DATA_PATH)
print(f"shape: {df.shape}")
df.head()

## Data Cleaning

Checking for nulls, duplicates, and basic stats before diving into analysis.

In [ ]:
# missing values
print("nulls:", df.isnull().sum().sum())

# duplicates
dup = df.duplicated().sum()
print(f"duplicates: {dup}")
if dup:
    df = df.drop_duplicates()
    print(f"after drop: {df.shape}")

# quick stats
df.describe()

In [ ]:
# class breakdown
counts = df["Class"].value_counts()
pcts = df["Class"].value_counts(normalize=True) * 100
print(f"Legitimate: {counts[0]:,} ({pcts[0]:.2f}%)")
print(f"Fraud:      {counts[1]:,} ({pcts[1]:.4f}%)")
print(f"Ratio: {pcts[0]/pcts[1]:.1f}:1")

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
colors = ["#2ecc71", "#e74c3c"]

# pie
counts.plot.pie(ax=axes[0,0], autopct="%1.2f%%", colors=colors, startangle=90,
                explode=(0, 0.05), labels=["Legitimate", "Fraud"])
axes[0,0].set_title("Class Distribution")
axes[0,0].set_ylabel("")

# bar
bars = axes[0,1].bar(["Legitimate", "Fraudulent"], counts.values, color=colors, width=0.5)
axes[0,1].set_title("Transaction Counts")
for bar, val in zip(bars, counts.values):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                   f"{val:,}", ha="center")

# amount distribution
for cls, color, label in zip([0, 1], colors, ["Legitimate", "Fraud"]):
    sns.histplot(df[df["Class"] == cls]["Amount"], ax=axes[0,2],
                 color=color, label=label, bins=50, alpha=0.6, kde=True)
axes[0,2].set_title("Transaction Amount")
axes[0,2].set_xlim(-10, 500)
axes[0,2].legend()

# time distribution
for cls, color, label in zip([0, 1], colors, ["Legitimate", "Fraud"]):
    sns.histplot(df[df["Class"] == cls]["Time"], ax=axes[1,0],
                 color=color, label=label, bins=50, alpha=0.6)
axes[1,0].set_title("Transaction Time")
axes[1,0].legend()

# amount boxplot
df.boxplot(column="Amount", by="Class", ax=axes[1,1], patch_artist=True,
           boxprops=dict(facecolor="#3498db"))
axes[1,1].set_title("Amount by Class")

# fraud rate by amount bin
df["AmountBin"] = pd.cut(df["Amount"], bins=[-1, 10, 50, 100, 200, 500, 10000],
                         labels=["0-10", "10-50", "50-100", "100-200", "200-500", "500+"])
fraud_rate = (df[df["Class"]==1]["AmountBin"].value_counts() /
              df["AmountBin"].value_counts() * 100).sort_index()
fraud_rate.plot(kind="bar", ax=axes[1,2], color="#e74c3c")
axes[1,2].set_title("Fraud Rate by Amount Range")
axes[1,2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# correlation heatmap
plt.figure(figsize=(16, 12))
corr = df.drop(columns=["AmountBin"]).corr()
sns.heatmap(corr, cmap="RdBu_r", center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title("Feature Correlations")
plt.tight_layout()
plt.show()

# pca feature distributions
fig, axes = plt.subplots(4, 7, figsize=(24, 14))
fig.suptitle("PCA Features (V1-V28) by Class", fontsize=16, y=1.02)
for i, col in enumerate([f"V{i}" for i in range(1, 29)]):
    ax = axes.flatten()[i]
    for cls, color, label in zip([0, 1], ["#2ecc71", "#e74c3c"], ["Legit", "Fraud"]):
        sns.kdeplot(df[df["Class"]==cls][col], ax=ax, color=color,
                    label=label, fill=True, alpha=0.3, linewidth=0.8)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=6)
for j in range(28, len(axes.flatten())):
    fig.delaxes(axes.flatten()[j])
plt.tight_layout()
plt.show()

df.drop(columns=["AmountBin"], inplace=True)

## Preprocessing & Feature Scaling

Splitting into train/test and scaling the non-PCA features.

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# tried StandardScaler first but it got thrown off by outliers in Amount
# sscaler = StandardScaler()
# X_train_s = sscaler.fit_transform(X_train[["Time", "Amount"]])
# the scaling was too sensitive to extreme values, switching to RobustScaler

cols_to_scale = ["Time", "Amount"]
scaler = RobustScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

X_train_scaled[cols_to_scale].describe()

## Handling Class Imbalance with SMOTE

Applying SMOTE on the training set only (no leakage into test). Using 0.5 sampling strategy so the minority class ends up at 50% of the majority.

In [ ]:
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Before - Legit: {(y_train==0).sum():,}, Fraud: {(y_train==1).sum():,}")
print(f"After  - Legit: {(y_train_resampled==0).sum():,}, Fraud: {(y_train_resampled==1).sum():,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, vals, title in [
    (ax1, [(y_train==0).sum(), (y_train==1).sum()], "Before SMOTE"),
    (ax2, [(y_train_resampled==0).sum(), (y_train_resampled==1).sum()], "After SMOTE")
]:
    ax.bar(["Legitimate", "Fraud"], vals, color=["#2ecc71", "#e74c3c"], width=0.5)
    ax.set_title(title)
    for i, v in enumerate(vals):
        ax.text(i, v + 500, f"{v:,}", ha="center")
plt.tight_layout()
plt.show()

## Model Training & Comparison

Training 6 models on the resampled data and evaluating on the original test set.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced"),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                              random_state=42, n_jobs=-1, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                                random_state=42, n_jobs=-1, verbose=-1, class_weight="balanced"),
    "CatBoost": CatBoostClassifier(n_estimators=100, learning_rate=0.1, depth=6,
                                    random_state=42, verbose=0),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42, class_weight="balanced")
}

results = []
trained_models = {}
conf_matrices = {}
roc_data = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_resampled, y_train_resampled)
    t = time.time() - t0

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred, pos_label=1),
        "Recall": recall_score(y_test, y_pred, pos_label=1),
        "F1-Score": f1_score(y_test, y_pred, pos_label=1),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Avg Precision": average_precision_score(y_test, y_prob),
        "Train Time (s)": round(t, 3)
    })
    trained_models[name] = model
    conf_matrices[name] = confusion_matrix(y_test, y_pred)
    roc_data[name] = (y_test, y_prob)

    print(f"{name:22s} | F1: {results[-1]['F1-Score']:.4f} | ROC: {results[-1]['ROC-AUC']:.4f} | {t:.2f}s")

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
results_df

### Model Comparison Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# metrics comparison
plot_data = results_df.melt(id_vars=["Model"],
    value_vars=["Precision", "Recall", "F1-Score", "ROC-AUC"],
    var_name="Metric", value_name="Score")
sns.barplot(data=plot_data, x="Model", y="Score", hue="Metric", ax=axes[0])
axes[0].set_title("Metrics by Model")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_ylim(0, 1.05)

# training time
time_df = results_df[["Model", "Train Time (s)"]].set_index("Model")
axes[1].barh(time_df.index, time_df["Train Time (s)"],
             color=["#e74c3c" if v > time_df["Train Time (s)"].median() else "#3498db"
                    for v in time_df["Train Time (s)"]])
axes[1].set_title("Training Time (seconds)")
for i, v in enumerate(time_df["Train Time (s)"]):
    axes[1].text(v + 0.1, i, f"{v:.2f}s", va="center", fontsize=9)

plt.tight_layout()
plt.show()

# confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Confusion Matrices", fontsize=14)
for i, (name, cm) in enumerate(conf_matrices.items()):
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit", "Fraud"]).plot(
        ax=axes.flatten()[i], cmap="Blues", colorbar=False, values_format="d")
    axes.flatten()[i].set_title(name, fontsize=11)
if len(conf_matrices) < 6:
    fig.delaxes(axes.flatten()[-1])
plt.tight_layout()
plt.show()

# ROC curves
plt.figure(figsize=(10, 8))
for name, (y_true, y_prob) in roc_data.items():
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {roc_auc_score(y_true, y_prob):.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# PR curves
plt.figure(figsize=(10, 8))
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    p, r, _ = precision_recall_curve(y_test, y_prob)
    plt.plot(r, p, linewidth=2, label=f"{name} (AP = {average_precision_score(y_test, y_prob):.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning

Running GridSearchCV on the best model from the comparison above. Using 3-fold stratified CV with F1 scoring.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print(f"Best model: {best_model_name}")

param_grids = {
    "Logistic Regression": {"C": [0.01, 0.1, 1, 10], "solver": ["lbfgs", "saga"]},
    "Random Forest": {"n_estimators": [100, 200], "max_depth": [10, 15, None],
                       "min_samples_split": [2, 5]},
    "XGBoost": {"n_estimators": [100, 200, 300], "learning_rate": [0.01, 0.05, 0.1],
                 "max_depth": [4, 6, 8], "subsample": [0.8, 1.0]},
    "LightGBM": {"n_estimators": [100, 200], "learning_rate": [0.01, 0.05, 0.1],
                  "max_depth": [4, 6, -1], "num_leaves": [15, 31]},
    "CatBoost": {"iterations": [100, 200], "learning_rate": [0.01, 0.05, 0.1],
                  "depth": [4, 6, 8]},
    "SVM": {"C": [0.1, 1, 10], "gamma": ["scale", "auto"], "kernel": ["rbf"]}
}

grid = GridSearchCV(
    models[best_model_name],
    param_grids[best_model_name],
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

# SVM is slow, sample the data if needed
if best_model_name == "SVM":
    from sklearn.utils import resample
    X_samp, y_samp = resample(X_train_resampled, y_train_resampled,
                               n_samples=int(0.3 * len(X_train_resampled)),
                               random_state=42, stratify=y_train_resampled)
    grid.fit(X_samp, y_samp)
else:
    grid.fit(X_train_resampled, y_train_resampled)

print(f"Best params: {grid.best_params_}")
print(f"Best CV F1: {grid.best_score_:.4f}")

### Best Model Evaluation

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_scaled)
y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

prec = precision_score(y_test, y_pred, pos_label=1)
rec = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)
roc = roc_auc_score(y_test, y_prob)

print(f"Tuned {best_model_name}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {roc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

# confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                        display_labels=["Legitimate", "Fraud"]).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Confusion Matrix - Tuned {best_model_name}")
plt.tight_layout()
plt.show()

# roc curve
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(best_model, X_test_scaled, y_test, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_title(f"ROC Curve - Tuned {best_model_name}")
ax.legend()
plt.tight_layout()
plt.show()

# feature importance if available
if hasattr(best_model, "feature_importances_"):
    imp = pd.DataFrame({"Feature": X_train.columns,
                         "Importance": best_model.feature_importances_})
    imp = imp.sort_values("Importance", ascending=True).tail(15)
    plt.figure(figsize=(10, 6))
    plt.barh(imp["Feature"], imp["Importance"], color="#3498db")
    plt.xlabel("Feature Importance")
    plt.title(f"Top 15 Features - Tuned {best_model_name}")
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, "coef_"):
    coef = pd.DataFrame({"Feature": X_train.columns,
                          "Coefficient": best_model.coef_.flatten()})
    coef["Abs"] = coef["Coefficient"].abs()
    coef = coef.sort_values("Abs", ascending=True).tail(15)
    plt.figure(figsize=(10, 6))
    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in coef["Coefficient"]]
    plt.barh(coef["Feature"], coef["Coefficient"], color=colors)
    plt.axvline(x=0, color="black", linewidth=0.5)
    plt.xlabel("Coefficient")
    plt.title(f"Top 15 Coefficients - Tuned {best_model_name}")
    plt.tight_layout()
    plt.show()

## Saving the Model

In [ ]:
MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(best_model, os.path.join(MODEL_DIR, "fraud_model.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))

# save config for reference
config = {
    "best_model": best_model_name,
    "best_params": grid.best_params_,
    "f1_score": float(f1),
    "roc_auc": float(roc)
}
with open(os.path.join(MODEL_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("saved to", MODEL_DIR)
for f in os.listdir(MODEL_DIR):
    sz = os.path.getsize(os.path.join(MODEL_DIR, f))
    print(f"  {f} ({sz/1024:.1f} KB)")

### Quick Inference Check

In [ ]:
def load_model(model_dir="saved_models"):
    m = joblib.load(os.path.join(model_dir, "fraud_model.pkl"))
    s = joblib.load(os.path.join(model_dir, "scaler.pkl"))
    return m, s

loaded, _ = load_model()

# test on a few samples
samples = X_test_scaled.iloc[:5]
actuals = y_test.iloc[:5]
preds = loaded.predict(samples)
probs = loaded.predict_proba(samples)[:, 1]

for i in range(5):
    pred_label = "FRAUD" if preds[i] else "legit"
    actual_label = "FRAUD" if actuals.iloc[i] else "legit"
    conf = probs[i] if preds[i] else 1 - probs[i]
    match = "OK" if preds[i] == actuals.iloc[i] else "MISMATCH"
    print(f"  {i+1}: predicted={pred_label} ({conf:.1%})  actual={actual_label}  [{match}]")
print("done")

In [ ]:
from IPython.display import display, Markdown

chosen = best_model_name
summary = ""

summary += "## Summary\n\n"
summary += "Trained and compared 6 classifiers on the Credit Card Fraud Detection dataset\n"
summary += "(284k transactions, 0.17% fraud rate). The main challenge was the extreme class\n"
summary += "imbalance, handled with SMOTE.\n\n"

summary += f"**Best model: {chosen}**  \n\n"
summary += "| Metric | Score |\n|--------|-------|\n"
summary += f"| F1-Score | {f1:.4f} |\n"
summary += f"| ROC-AUC | {roc:.4f} |\n"
summary += f"| Precision | {prec:.4f} |\n"
summary += f"| Recall | {rec:.4f} |\n\n"

summary += "**What worked:**\n"
summary += "- SMOTE made a big difference - without it recall was much worse\n"
summary += "- Gradient boosting models (XGBoost/LightGBM/CatBoost) consistently outperformed LR and RF\n"
summary += "- RobustScaler handled the Amount outliers better than StandardScaler\n\n"

summary += "**What I'd try next:**\n"
summary += "- Threshold tuning - the default 0.5 isnt optimal for fraud detection\n"
summary += "- Deeper hyperparameter search on the boosting models\n"
summary += "- Anomaly detection approaches (Isolation Forest, autoencoders)\n"
summary += "- Proper cross-validation with SMOTE inside each fold\n"
summary += "- Deployment as a simple API for real-time scoring\n\n"

summary += "---\n"
summary += "*Built with Python, scikit-learn, XGBoost, LightGBM, CatBoost, and SMOTE.*"

display(Markdown(summary))